# Classifying Engine Faults

### Import Packages

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

from matplotlib.colors import ListedColormap
from scipy.stats import ttest_rel, wilcoxon
from sklearn.datasets import make_regression
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import Lasso, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor


### EDA

In [ ]:
engines = pd.read_csv('EngineFaultDB_Final.csv')

In [ ]:
engines.describe()

In [ ]:
engines.dtypes

In [ ]:
engines.shape

In [ ]:
engines.isnull().sum() # no missing data

In [ ]:
engines.Fault.unique()

In [ ]:
s = engines["Fault"].dropna()

# discrete values (e.g., 1..4)
vals = sorted(s.unique())
counts = s.value_counts().reindex(vals).fillna(0).astype(int)

# percent out of all rows (not cumulative)
pct = counts / counts.sum() * 100

# histogram-like bars (counts) + percentage labels/line on same x
bins = [v - 0.5 for v in vals] + [vals[-1] + 0.5]

sns.set_theme(style="white", palette="Set2")
bar_color = sns.color_palette("Set2")[2]
line_color = sns.color_palette("Set2")[1]

fig, ax1 = plt.subplots(figsize=(8, 4.5))

# histogram bars (counts)
ax1.hist(s, bins=bins, edgecolor="black", alpha=0.85, color=bar_color)
ax1.set_xticks(vals)
ax1.set_xlabel("Fault")
ax1.set_ylabel("Count")

# percentage line (percent of all rows)
ax2 = ax1.twinx()
ax2.plot(vals, pct.values, marker="o", linewidth=2, color=line_color)
ax2.set_ylabel("Percent of All Rows")
ax2.set_ylim(0, max(10, float(np.ceil(pct.max()/5)*5)))  # reasonable top

ax1.set_title("Fault Histogram & Percent of All Rows")
ax1.grid(True, alpha=0.3)

plt.show()


In [ ]:
for col in engines.columns:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Histogram
    sns.histplot(engines[col].dropna(), kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title(f'Histogram of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frequency')
    
    # Boxplot
    sns.boxplot(x=engines[col], ax=axes[1], color='lightgreen')
    axes[1].set_title(f'Boxplot of {col}')
    axes[1].set_xlabel(col)
    
    plt.tight_layout()
    plt.show()

In [ ]:
corr = engines.corr()
plt.figure(figsize=(8,5))
plt.title('Heatmap of Variable Correlation')
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.show()

In [ ]:
# use VIF to check multicollinearity
# add constant for statsmodels
X_const = sm.add_constant(engines.iloc[:,1:])

vif_df_feature_selection = pd.DataFrame()
vif_df_feature_selection["variable"] = X_const.columns
vif_df_feature_selection["VIF"] = [variance_inflation_factor(X_const.values, i)
                 for i in range(X_const.shape[1])]
vif_df_feature_selection = vif_df_feature_selection[vif_df_feature_selection["variable"] != "const"]
print(vif_df_feature_selection.sort_values("VIF", ascending=False).head())

# VIF<3 -> no multicollinearity
# VIF 3-5 -> moderate
# VIF>5 -> high
def vif_categorizer(x):
    if x<=1:
        return 'none'
    if x>1 and x<=5:
        return 'moderate'
    if x>5:
        return 'high'
vif_df_feature_selection['multicollinearity'] = vif_df_feature_selection['VIF'].apply(lambda x: vif_categorizer(x))
vif_df_feature_selection.sort_values('VIF', ascending=False)

### Modeling

In [ ]:
# generate splits
X = engines.iloc[:,1:]
y = engines.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
# ridge regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
# lasso regression
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
df_feature_selection = pd.DataFrame()
df_feature_selection['Variable'] = X.columns
df_feature_selection["Ridge Coefficients"] = ridge.coef_
df_feature_selection['LASSO Coefficients'] = lasso.coef_

In [ ]:
df_feature_selection.sort_values(by='Ridge Coefficients')
# deciding on ridge to avoid sparsity
#.1 * max(ridge coeff) = .142 -> used as threshold to keep variables

In [ ]:
selected_features = df_feature_selection[(df_feature_selection['Ridge Coefficients']>.142) | (df_feature_selection['Ridge Coefficients']<-.142)].Variable

In [ ]:
# engine characteristics only X
engine_characteristics = ['MAP', 'TPS', 'Force', 'Power', 'RPM', 'Consumption L/H', 'Consumption L/100KM', 'Speed']
engines_1 = engines[engine_characteristics]

# exhaust only X
exhaust_characteristics = ['CO', 'HC', 'CO2', 'O2', 'Lambda', 'AFR']
engines_2 = engines[exhaust_characteristics]

# variable selected X
engines_3 = engines[selected_features]

In [ ]:
def model_results(X, best_k):
    Ni = X.shape[1]      # number of input features (should be 14)
    No = len(np.unique(y))  # number of output classes (should be 4)
    Nh = int(0.5 * (Ni + No))  # Piramuthu rule → 9 neurons

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

    #  BEST-K SEARCH FOR KNN (K = 1 → 25)
    best_k = None
    best_k_score = -1
    for k in range(1, 26):
        knn_model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=k))
        ])
        scores = cross_val_score(knn_model, X, y, cv=cv, scoring="accuracy")
        mean_score = scores.mean()

        if mean_score > best_k_score:
            best_k_score = mean_score
            best_k = k

    print(f"Best K found: {best_k} with accuracy={best_k_score:.4f}")

    #  FEED-FORWARD NN USING PURE SCIKIT-LEARN (MLPClassifier)
    ffnn_clf = MLPClassifier(
        hidden_layer_sizes=(Nh,),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=100,
        batch_size=64,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=123
    )

    #  MODELS DICTIONARY
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=500))
        ]),
        "Decision Tree": DecisionTreeClassifier(random_state=123),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=123),
        "SVC": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=False, random_state=123))
        ]),
        "Best-K KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=best_k))
        ]),
        "Naive Bayes": GaussianNB(),
        "Feed-Forward NN (Piramuthu Rule)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", ffnn_clf)
        ]),
        "QDA": QuadraticDiscriminantAnalysis(),
        "LDA": LinearDiscriminantAnalysis(),
        "Gradient Boosting": GradientBoostingClassifier(random_state=123)
    }

    #  SCORING METRICS
    scoring = {
        "accuracy": "accuracy",
        "precision": 'precision_macro',
        "recall": 'recall_macro',
        "f1": 'f1_macro'
    }

    #  CROSS-VALIDATION FOR ALL MODELS
    results = {}
    results_long = {}

    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=True)
        results[name] = {
            "train_accuracy": scores["train_accuracy"].mean(),
            "accuracy": scores["test_accuracy"].mean(),
            "precision": scores["test_precision"].mean(),
            "recall": scores["test_recall"].mean(),
            "f1": scores["test_f1"].mean()
        }
        results_long[name] = scores['test_f1']

        print(name, ' done')
    #  PRINT RESULTS
        for name, metrics in results.items():
            print(f"\n{name}")
            print(f"  Train Acc : {metrics['train_accuracy']:.4f}")
            print(f"  Val Acc   : {metrics['accuracy']:.4f}")
            print(f"  Precision : {metrics['precision']:.4f}")
            print(f"  Recall    : {metrics['recall']:.4f}")
            print(f"  F1        : {metrics['f1']:.4f}")
    return results, results_long

engines_1_results, engines_1_results_long = model_results(engines_1,4)
engines_2_results, engines_2_results_long = model_results(engines_2,3)
engines_3_results, engines_3_results_long = model_results(engines_3,23)

In [ ]:
with open("engines_3_full_results.txt", "w") as f:
    for model_name in engines_3_results:
        f.write(f"Model: {model_name}\n")

        # Average metrics
        for metric, value in engines_3_results[model_name].items():
            f.write(f"  {metric}: {value:.4f}\n")

        # Fold-level metrics
        f.write("  F1 per fold:\n")
        for i, score in enumerate(engines_3_results_long[model_name], start=1):
            f.write(f"    Fold {i}: {score:.4f}\n")

        f.write("\n")


In [ ]:
def write_engine_results(engine_number):
    # Dynamically select the correct dictionaries
    avg_results = globals()[f"engines_{engine_number}_results"]
    long_results = globals()[f"engines_{engine_number}_results_long"]

    # 1. Write average results
    with open(f"engines_{engine_number}_average_results.txt", "w") as f:
        for model_name, metrics in avg_results.items():
            f.write(f"Model: {model_name}\n")
            for metric, value in metrics.items():
                f.write(f"  {metric}: {value:.4f}\n")
            f.write("\n")

    # 2. Write fold-level results
    with open(f"engines_{engine_number}_fold_results.txt", "w") as f:
        for model_name, folds in long_results.items():
            f.write(f"{model_name} F1 scores per fold:\n")
            for i, score in enumerate(folds, start=1):
                f.write(f"  Fold {i}: {score:.4f}\n")
            f.write("\n")

    # 3. Wilcoxon + FDR significance
    pairs = list(itertools.combinations(long_results.keys(), 2))
    pvals = []

    for a, b in pairs:
        stat, p = wilcoxon(long_results[a], long_results[b])
        pvals.append(p)

    reject, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

    with open(f"engines_{engine_number}_wilcoxon_results.txt", "w") as f:
        for (a, b), p, pc, r in zip(pairs, pvals, pvals_corrected, reject):
            f.write(f"{a} vs {b}:\n")
            f.write(f"  raw p-value:       {p:.6f}\n")
            f.write(f"  FDR corrected p:   {pc:.6f}\n")
            f.write(f"  significant (FDR): {r}\n")
            f.write("\n")

    print(f"Finished writing results for engines_{engine_number}")


In [ ]:
write_engine_results(1)
write_engine_results(2)
write_engine_results(3)

In [ ]:
def wilcoxon_heatmap_all_models(eng1, eng2, eng3, alpha=0.10):
    models = list(eng1.keys())

    custom_abbrev = {
        "GradientBoosting": "GB",
        "GradientBoostingClassifier": "GB",
        "NeuralNet": "FFNN",
        "MLPClassifier": "FFNN",
        "NaiveBayes": "NB",
        "GaussianNB": "NB",
        "LogisticRegression": "LR",
        "DecisionTree": "DT",
        "DecisionTreeClassifier": "DT",
        "RandomForest": "RF",
        "RandomForestClassifier": "RF",
        "SVC": "SVC",
        "SupportVectorClassifier": "SVC",
        "KNN": "KNN",
        "KNeighborsClassifier": "KNN"
    }

    abbrev = {m: custom_abbrev.get(m, m.upper()) for m in models}

    comparisons = ["Engine vs Gas", "Engine vs VS", "Gas vs VS"]
    heatmap_data = pd.DataFrame(index=[abbrev[m] for m in models],
                                columns=comparisons)

    for model in models:
        data = {
            "Engine": eng1[model],
            "Gas": eng2[model],
            "VS": eng3[model]
        }

        pairs = [("Engine", "Gas"),
                 ("Engine", "VS"),
                 ("Gas", "VS")]

        raw_pvals = []
        for a, b in pairs:
            _, p = wilcoxon(data[a], data[b])
            raw_pvals.append(p)

        reject, corrected, _, _ = multipletests(raw_pvals, alpha=alpha, method='fdr_bh')
        heatmap_data.loc[abbrev[model], :] = corrected

    light_blue = "#a0cbf4"   # p >= 0.10
    orange     = "#ebb68b"   # p < 0.10

    cmap = ListedColormap([light_blue, orange])

    binary_mask = (heatmap_data.astype(float) < 0.10).astype(int)

    plt.figure(figsize=(10, 6))
    sns.heatmap(
        binary_mask,
        annot=heatmap_data.astype(float).round(4),
        fmt=".4f",
        cmap=cmap,
        linewidths=0.5,
        cbar=False
    )

    plt.title("Wilcoxon Significance Across All Models")
    plt.xlabel("Engine Comparison")
    plt.ylabel("Model")
    plt.tight_layout()
    plt.show()

    return heatmap_data


In [ ]:
wilcoxon_matrix = wilcoxon_heatmap_all_models(
    engines_1_results_long,
    engines_2_results_long,
    engines_3_results_long,
    alpha=0.10
)

print(wilcoxon_matrix)


In [ ]:
def plot_cv_distributions_separate(eng1, eng2, eng3, baseline=0.751):
    engines = {
        "Engine-Only": eng1,
        "Gas-Only": eng2,
        "Variable Selection": eng3
    }

    palette = sns.color_palette("Set2", 1)[0]

    fig, axes = plt.subplots(3, 1, figsize=(10,15), sharey=True)

    for ax, (engine_name, eng_data) in zip(axes, engines.items()):
        rows = []
        for model, scores in eng_data.items():
            for s in scores:
                rows.append({"Model": model, "F1": s})

        df = pd.DataFrame(rows)

        # --- filter out models with mean F1 < 0.7 ---
        model_means = df.groupby("Model")["F1"].mean()
        keep_models = model_means[model_means >= 0.7].index
        df = df[df["Model"].isin(keep_models)]

        sns.boxplot(
            data=df,
            x="Model",
            y="F1",
            color=palette,
            ax=ax
        )

        # orange baseline line
        ax.axhline(baseline, color="orange", linestyle="--", linewidth=2)

        ax.set_title(f"Cross-Validation F1 — {engine_name}")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45)


    fig.tight_layout()
    plt.show()


In [ ]:
plot_cv_distributions_separate(
    engines_1_results_long,
    engines_2_results_long,
    engines_3_results_long,
    baseline=0.751
)